# Predicting Voter Turnout from European Social Survey Round 11

**Course:** Machine Learning and Deep Learning, Spring 2026 (CBS)
**Authors:** Andreja + group (4)
**Dataset:** ESS11e04_1.csv (European Social Survey Round 11, integrated edition 4.1)
**Target:** `vote` — did the respondent vote in the last national election? (binary)

This notebook is the working analysis. The project plan lives in `GAMEPLAN.md`; the per-step explanations live in `METHODOLOGY_DETAILED.md`; the reflection lives in `SELF_IMPROVEMENT.md`.

Section structure mirrors the report template:
1. Setup & data load
2. Variable selection (the 5 conceptual blocks)
3. Sentinel-code recoding & target filtering
4. EDA
5. Preprocessing pipeline + train / val / test split
6. Models — Logistic Regression (Lec 04), Random Forest (Lec 05), Gradient Boosting (Lec 05), Feed-Forward Neural Network (Lec 08–09)
7. Evaluation
8. Interpretation
9. Cross-country error analysis


## 1. Setup & data load

In [3]:
# Standard ML/DL stack used throughout the course
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 160)
sns.set_theme(style='whitegrid', context='notebook')

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

DATA_PATH = Path('ESS11e04_1.csv')  # expected to live next to this notebook
assert DATA_PATH.exists(), f'Place ESS11e04_1.csv next to this notebook. Looking at: {DATA_PATH.resolve()}'


AssertionError: Place ESS11e04_1.csv next to this notebook. Looking at: /Users/andrejavasic/Desktop/ML & DL/Machine Learning/ESS11e04_1.csv

In [ ]:
# Load — full file is ~50k rows x 691 cols; low_memory=False avoids dtype warnings
df_raw = pd.read_csv(DATA_PATH, low_memory=False)
print(f'Shape: {df_raw.shape}')
df_raw.head()


## 2. Variable selection — the 5 conceptual blocks

We work with ~25 columns derived from five conceptual blocks plus the country indicator.
See `GAMEPLAN.md` §2.1 for the theoretical justification.

In [ ]:
# The five conceptual blocks (column lists). Edit here if the team decides to add/drop a variable.

TARGET = 'vote'

VARS = {
    'demo':       ['agea', 'gndr', 'cntry', 'domicil', 'brncntr'],
    'econ':       ['eisced', 'hinctnta', 'mnactic', 'uemp12m'],
    'polint':     ['polintr', 'lrscale'],
    'trust':      ['trstplt', 'trstprl', 'trstprt', 'trstlgl', 'trstplc',
                   'stfdem', 'stfgov'],
    'civic':      ['rlgatnd', 'sclmeet', 'mbtru', 'nwspol', 'sclact'],
}
ALL_FEATURES = [v for block in VARS.values() for v in block]
print(f'Total features (incl. cntry): {len(ALL_FEATURES)}')

# Verify every column exists in the raw data — fail loudly if a name was renamed in this round
missing = [c for c in [TARGET] + ALL_FEATURES if c not in df_raw.columns]
assert not missing, f'Missing from data: {missing}'


In [ ]:
# Reduce to the columns we'll actually use, plus survey weights for descriptives
KEEP = [TARGET] + ALL_FEATURES + [c for c in ['dweight', 'pspwght'] if c in df_raw.columns]
df = df_raw[KEEP].copy()
print(df.shape)
df.dtypes


## 3. Sentinel-code recoding & target filtering

ESS uses numeric sentinel codes (e.g. 77/88/99, 7/8/9, 6666/7777/8888/9999) for refusal / don't know / no answer / not applicable. They must be converted to `NaN` *before* any analysis — otherwise a code like 99 will be silently treated as a real value.

The exact code set varies by variable (see `GAMEPLAN.md` §3.4 for the full table).

In [ ]:
# Per-variable sentinel maps. None of the substantive answers below ever touch these values.
SENTINELS = {
    'vote':     [7, 8, 9],        # 3 ('not eligible') is filtered separately below
    'agea':     [999],
    'gndr':     [9],
    'domicil':  [7, 8, 9],
    'brncntr':  [7, 8, 9],
    'eisced':   [55, 77, 88, 99],
    'hinctnta': [77, 88, 99],
    'mnactic':  [66, 77, 88, 99],
    'uemp12m':  [6, 7, 8, 9],
    'polintr':  [7, 8, 9],
    'lrscale':  [77, 88, 99],
    'trstplt':  [77, 88, 99],
    'trstprl':  [77, 88, 99],
    'trstprt':  [77, 88, 99],
    'trstlgl':  [77, 88, 99],
    'trstplc':  [77, 88, 99],
    'stfdem':   [77, 88, 99],
    'stfgov':   [77, 88, 99],
    'rlgatnd':  [77, 88, 99],
    'sclmeet':  [77, 88, 99],
    'mbtru':    [7, 8, 9],
    'nwspol':   [6666, 7777, 8888, 9999],
    'sclact':   [7, 8, 9],
}

for col, codes in SENTINELS.items():
    df[col] = df[col].replace(codes, np.nan)


In [ ]:
# Filter target: keep only voters (1) and non-voters (2). Drop 'not eligible' (3) and missings.
print('vote raw value counts (before filter):')
print(df['vote'].value_counts(dropna=False).sort_index())

df = df[df['vote'].isin([1, 2])].copy()
df['vote'] = (df['vote'] == 1).astype(int)  # 1 = voted, 0 = did not vote

print()
print(f'After filtering, n = {len(df):,}; voted share = {df["vote"].mean():.1%}')


In [ ]:
# Reverse-code two ordinal variables so 'higher = more' across the board.
# polintr in ESS is coded 1=very interested .. 4=not at all interested → reverse so higher = more interested
df['polintr'] = 5 - df['polintr']

# rlgatnd is coded 1=every day .. 7=never → reverse so higher = attends more often
df['rlgatnd'] = 8 - df['rlgatnd']

# Recode binary variables to 0/1 with informative direction
df['gndr']    = (df['gndr'] == 2).astype('Int64')        # 1 = female
df['brncntr'] = (df['brncntr'] == 2).astype('Int64')     # 1 = born abroad
df['mbtru']   = (df['mbtru'] == 1).astype('Int64')       # 1 = current trade-union member
df['uemp12m'] = (df['uemp12m'] == 1).astype('Int64')     # 1 = was unemployed >12m


## 4. Exploratory Data Analysis

We produce, in order:
1. Target distribution overall and by country
2. Missingness heat-map (after recoding!)
3. Univariate distributions of features
4. Bivariate plots vs. `vote`
5. Spearman correlation among numeric features


In [ ]:
# 4.1 Target distribution by country
turnout_by_country = (
    df.groupby('cntry')['vote']
      .agg(['mean', 'count'])
      .sort_values('mean', ascending=False)
)
turnout_by_country.columns = ['turnout_share', 'n']
turnout_by_country


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
turnout_by_country['turnout_share'].plot(kind='barh', ax=ax)
ax.set_xlabel('Self-reported turnout share')
ax.set_title('Self-reported turnout by country (ESS11)')
ax.axvline(df['vote'].mean(), ls='--', color='black', label='pooled mean')
ax.legend()
plt.tight_layout()


In [ ]:
# 4.2 Missingness — AFTER sentinel recoding
miss = df[ALL_FEATURES].isna().mean().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(8, 6))
miss.plot(kind='barh', ax=ax, color='tomato')
ax.set_xlabel('Share missing')
ax.set_title('Missingness per feature (post sentinel-recoding)')
plt.tight_layout()
miss.to_frame('missing_share')


In [ ]:
# 4.3 Univariate distributions for numeric / ordinal features
numeric_like = ['agea', 'eisced', 'hinctnta', 'lrscale',
                'trstplt', 'trstprl', 'trstprt', 'trstlgl', 'trstplc',
                'stfdem', 'stfgov',
                'polintr', 'rlgatnd', 'sclmeet', 'sclact', 'nwspol']
df[numeric_like].hist(bins=30, figsize=(14, 10))
plt.tight_layout()


In [ ]:
# 4.4 Bivariate vs. vote — turnout share by ordinal/categorical bin
def turnout_by(feature, ax):
    s = df.groupby(feature)['vote'].mean()
    s.plot(kind='bar', ax=ax)
    ax.set_ylabel('turnout share')
    ax.set_title(feature)
    ax.set_ylim(0, 1)

cols_to_plot = ['eisced', 'hinctnta', 'polintr', 'lrscale',
                'trstplt', 'stfdem', 'rlgatnd', 'sclmeet']
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for col, ax in zip(cols_to_plot, axes.flat):
    turnout_by(col, ax)
plt.tight_layout()


In [ ]:
# 4.5 Spearman correlations among numeric features (ordinal-safe)
import scipy.stats as stats
corr = df[numeric_like].corr(method='spearman')
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=ax)
ax.set_title('Spearman correlation, numeric/ordinal features')
plt.tight_layout()


## 5. Preprocessing pipeline + train / val / test split

We use a single `ColumnTransformer` so that imputation, encoding, and scaling are fit only on the training set — preventing leakage into validation and test sets.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Drop rows missing the target (already done) and rows with too many missing features
df['n_missing'] = df[ALL_FEATURES].isna().sum(axis=1)
df = df[df['n_missing'] <= len(ALL_FEATURES) // 2].drop(columns='n_missing')
print(f'After filtering on row-wise missingness: {len(df):,} rows')

# Engineering feature: log-transform nwspol (heavy right tail; minutes per day reading politics)
df['nwspol_log'] = np.log1p(df['nwspol'])

# Define column types for the pipeline
NUMERIC = ['agea', 'eisced', 'hinctnta', 'lrscale',
           'trstplt', 'trstprl', 'trstprt', 'trstlgl', 'trstplc',
           'stfdem', 'stfgov',
           'polintr', 'rlgatnd', 'sclmeet', 'sclact', 'nwspol_log']
BINARY  = ['gndr', 'brncntr', 'mbtru', 'uemp12m']
NOMINAL = ['cntry', 'domicil', 'mnactic']

X = df[NUMERIC + BINARY + NOMINAL].copy()
y = df['vote'].values

# Stratify on vote × country to preserve both class & country balance
strata = df['vote'].astype(str) + '_' + df['cntry'].astype(str)
X_temp, X_test, y_temp, y_test, strata_temp, _ = train_test_split(
    X, y, strata, test_size=0.15, stratify=strata, random_state=RANDOM_STATE)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.1765,  # 0.1765 of 0.85 ≈ 0.15 of original
    stratify=strata_temp, random_state=RANDOM_STATE)
print(f'train: {len(X_train):,} | val: {len(X_val):,} | test: {len(X_test):,}')
print(f'turnout train: {y_train.mean():.1%} | val: {y_val.mean():.1%} | test: {y_test.mean():.1%}')


In [ ]:
# Preprocessing pipeline
numeric_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('scale',  StandardScaler()),
])
binary_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='most_frequent')),
])
nominal_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='constant', fill_value='Missing')),
    ('ohe',    OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

preprocess = ColumnTransformer([
    ('num', numeric_pipe, NUMERIC),
    ('bin', binary_pipe,  BINARY),
    ('nom', nominal_pipe, NOMINAL),
])
preprocess


## 6. Models

Four model families, in increasing capacity. Every model is anchored to a specific course lecture so the choice can be defended at the oral exam.

- 6.1 Logistic Regression (Lec 04) — linear baseline, interpretable coefficients
- 6.2 Random Forest (Lec 05) — bagging ensemble, captures non-linearities
- 6.3 Gradient Boosting (Lec 05) — boosted ensemble, usually the strongest classical model on tabular data
- 6.4 Feed-Forward MLP (Lec 08–09) — deep-learning component; honest test of whether DL adds anything over GB

### 6.1 Logistic Regression (baseline)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                             average_precision_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)

lr_pipe = Pipeline([('prep', preprocess),
                    ('clf',  LogisticRegression(max_iter=2000,
                                                random_state=RANDOM_STATE))])
lr_pipe.fit(X_train, y_train)

def report(name, model, X_eval, y_eval):
    proba = model.predict_proba(X_eval)[:, 1]
    pred  = (proba >= 0.5).astype(int)
    return {
        'model':     name,
        'accuracy':  accuracy_score(y_eval, pred),
        'f1':        f1_score(y_eval, pred),
        'roc_auc':   roc_auc_score(y_eval, proba),
        'pr_auc':    average_precision_score(y_eval, proba),
    }

results = [report('Logistic Regression', lr_pipe, X_val, y_val)]
pd.DataFrame(results)


### 6.2 Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_pipe = Pipeline([('prep', preprocess),
                    ('clf',  RandomForestClassifier(
                        n_estimators=400, max_depth=None,
                        min_samples_leaf=20, n_jobs=-1,
                        random_state=RANDOM_STATE))])
rf_pipe.fit(X_train, y_train)

results.append(report('Random Forest', rf_pipe, X_val, y_val))
pd.DataFrame(results)


### 6.3 Gradient Boosting

Boosted ensemble of shallow trees (Lec 05). On tabular data of this size, gradient boosting is consistently the strongest classical model — typically beating random forests and often beating feed-forward neural networks too. We use scikit-learn's `HistGradientBoostingClassifier` to keep dependencies minimal.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier

gb_pipe = Pipeline([('prep', preprocess),
                    ('clf',  HistGradientBoostingClassifier(
                        learning_rate=0.05,
                        max_iter=400,
                        max_leaf_nodes=31,
                        min_samples_leaf=20,
                        random_state=RANDOM_STATE))])
gb_pipe.fit(X_train, y_train)

results.append(report('Gradient Boosting', gb_pipe, X_val, y_val))
pd.DataFrame(results)

### 6.4 Feed-Forward Neural Network (Keras)

The deep-learning component (Lec 08–09 build the MLP / backprop / dropout / Adam / early-stopping toolkit). We do **not** assume the MLP will beat gradient boosting — including it is the honest test of whether deep learning adds anything on top of the strongest classical model. Whether it wins or loses, that is a finding for the report.

In [ ]:
# Preprocess once for Keras (Keras expects pre-encoded numeric arrays)
X_train_p = preprocess.fit_transform(X_train)
X_val_p   = preprocess.transform(X_val)
X_test_p  = preprocess.transform(X_test)

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

tf.random.set_seed(RANDOM_STATE)

n_features = X_train_p.shape[1]

nn = models.Sequential([
    layers.Input(shape=(n_features,)),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(16, activation='relu'),
    layers.Dense(1,  activation='sigmoid'),
])
nn.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
           loss='binary_crossentropy',
           metrics=['accuracy', tf.keras.metrics.AUC(name='roc_auc')])
nn.summary()


In [ ]:
es = callbacks.EarlyStopping(monitor='val_loss', patience=8,
                              restore_best_weights=True)

history = nn.fit(X_train_p, y_train,
                 validation_data=(X_val_p, y_val),
                 epochs=80,
                 batch_size=256,
                 callbacks=[es],
                 verbose=2)

# Plot training curves
hist = pd.DataFrame(history.history)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
hist[['loss', 'val_loss']].plot(ax=axes[0], title='Loss')
hist[['roc_auc', 'val_roc_auc']].plot(ax=axes[1], title='ROC-AUC')
plt.tight_layout()


In [ ]:
# NN evaluation in the same shape as LR / RF
nn_proba_val = nn.predict(X_val_p, verbose=0).ravel()
nn_pred_val  = (nn_proba_val >= 0.5).astype(int)
results.append({
    'model':    'Neural Network',
    'accuracy': accuracy_score(y_val, nn_pred_val),
    'f1':       f1_score(y_val, nn_pred_val),
    'roc_auc':  roc_auc_score(y_val, nn_proba_val),
    'pr_auc':   average_precision_score(y_val, nn_proba_val),
})
pd.DataFrame(results)


## 7. Test-set evaluation (best model only)

Pick the best model on validation, then evaluate it once on the held-out test set.

In [ ]:
best = max(results, key=lambda r: r['roc_auc'])
print('Best model on validation:', best['model'])

if best['model'] == 'Logistic Regression':
    final = lr_pipe
    proba_test = final.predict_proba(X_test)[:, 1]
elif best['model'] == 'Random Forest':
    final = rf_pipe
    proba_test = final.predict_proba(X_test)[:, 1]
elif best['model'] == 'Gradient Boosting':
    final = gb_pipe
    proba_test = final.predict_proba(X_test)[:, 1]
else:
    proba_test = nn.predict(X_test_p, verbose=0).ravel()

pred_test = (proba_test >= 0.5).astype(int)

print()
print(classification_report(y_test, pred_test, target_names=['did not vote', 'voted']))
print('ROC-AUC :', roc_auc_score(y_test, proba_test).round(4))
print('PR-AUC  :', average_precision_score(y_test, proba_test).round(4))

ConfusionMatrixDisplay.from_predictions(y_test, pred_test,
                                       display_labels=['did not vote', 'voted'])
plt.title(f'Confusion matrix — {best["model"]}')
plt.show()


In [ ]:
# ROC and PR curves
from sklearn.metrics import roc_curve, precision_recall_curve

fpr, tpr, _ = roc_curve(y_test, proba_test)
prec, rec, _ = precision_recall_curve(y_test, proba_test)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(fpr, tpr); axes[0].plot([0,1],[0,1], '--', color='grey')
axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR'); axes[0].set_title('ROC')
axes[1].plot(rec, prec); axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall')
plt.tight_layout()


## 8. Interpretation — which variables matter?

We compare LR coefficients against RF permutation importance against (optionally) SHAP values.

In [ ]:
# 8.1 Logistic-regression coefficients — sign and magnitude
ohe_names = (lr_pipe.named_steps['prep']
                .named_transformers_['nom']
                .named_steps['ohe'].get_feature_names_out(NOMINAL))
feature_names = NUMERIC + BINARY + list(ohe_names)
coefs = pd.Series(lr_pipe.named_steps['clf'].coef_[0], index=feature_names)
top = coefs.abs().sort_values(ascending=False).head(20).index
fig, ax = plt.subplots(figsize=(8, 6))
coefs.loc[top].sort_values().plot(kind='barh', ax=ax)
ax.set_title('Top 20 LR coefficients (standardised features)')
plt.tight_layout()


In [ ]:
# 8.2 Random-forest permutation importance on validation set
from sklearn.inspection import permutation_importance

perm = permutation_importance(rf_pipe, X_val, y_val,
                              n_repeats=5, random_state=RANDOM_STATE,
                              n_jobs=-1)
imp = pd.Series(perm.importances_mean,
                index=NUMERIC + BINARY + NOMINAL).sort_values()
fig, ax = plt.subplots(figsize=(8, 6))
imp.tail(15).plot(kind='barh', ax=ax)
ax.set_title('Random Forest — permutation importance (top 15)')
plt.tight_layout()


## 9. Cross-country error analysis (RQ3)

Does the pooled model perform equally well across countries?

In [ ]:
country_test = X_test['cntry'].values
err = pd.DataFrame({'cntry': country_test,
                    'y_true': y_test,
                    'y_pred': pred_test,
                    'p':      proba_test})

per_country = err.groupby('cntry').apply(
    lambda g: pd.Series({
        'n':       len(g),
        'acc':     (g.y_true == g.y_pred).mean(),
        'roc_auc': roc_auc_score(g.y_true, g.p) if g.y_true.nunique() == 2 else np.nan,
    })
).sort_values('roc_auc')
per_country


## 10. Notes for the team

- The order of the cells matches `GAMEPLAN.md` and `METHODOLOGY_DETAILED.md`. Any change to the variable list, sentinel codes, or model architectures should be reflected in those documents too.
- All randomness is seeded by `RANDOM_STATE = 42`. Do not introduce new RNGs without seeding them — results must be reproducible for the oral exam.
- Hyperparameter tuning is left as a follow-up (`GridSearchCV` / `RandomizedSearchCV`). Add it once the pipeline runs end-to-end.
